In [1]:
import pandas as pd
from IPython.display import display

In [2]:
DATA_PATH_CH9 = "fraud_transactions.csv" # Path to the transaction data

try:
    transactions_df = pd.read_csv(DATA_PATH_CH9)
    print(f"'{DATA_PATH_CH9}' loaded successfully.")
    print("--------------------------------------------------")
    print("First 5 rows of the transaction dataset:")
    display(transactions_df.head())
    print("\n--------------------------------------------------")
    print(f"Dataset Shape: {transactions_df.shape}")
    print("--------------------------------------------------")
    print("Dataset Info:")
    transactions_df.info()
    print("\n--------------------------------------------------")
    if 'is_fraud' in transactions_df.columns:
        print("Distribution of 'is_fraud' label:")
        display(transactions_df['is_fraud'].value_counts(normalize=True).reset_index().rename(
            columns={'index': 'Is_Fraud', 'is_fraud': 'Proportion'}
        ))
    else:
        print("Note: 'is_fraud' column not found.")
    print("--------------------------------------------------")
except FileNotFoundError:
    print(f"Error: '{DATA_PATH_CH9}' not found. Please ensure the file is available.")
    transactions_df = None

'fraud_transactions.csv' loaded successfully.
--------------------------------------------------
First 5 rows of the transaction dataset:


,transaction_id,user_id,transaction_time,ip,device_id,phone_number,credit_card_number,order_item,amount,is_fraud
0,05987d33-9bcd-4455-9d7c-2f043b007b10,0,2024-05-07,219.20.206.172,8efe3214-ea7f-47ef-bb30-716e28f02600,(527)638-2221x33874,4908554227138584,Accessories,201631,False
1,013414be-3e37-48fb-aab2-5f42a4933db8,0,2024-05-09,219.20.206.172,8efe3214-ea7f-47ef-bb30-716e28f02600,(527)638-2221x33874,4908554227138584,Kids Products,281063,False
2,c6d1eed6-2c40-44f0-8a6f-fec0f8f7a83b,0,2024-05-12,219.20.206.172,8efe3214-ea7f-47ef-bb30-716e28f02600,(527)638-2221x33874,4908554227138584,Cosmetics,279119,False
3,90321501-3b4f-4cd3-92e9-ae1531b6b705,0,2024-05-01,219.20.206.172,8efe3214-ea7f-47ef-bb30-716e28f02600,(527)638-2221x33874,4908554227138584,Accessories,253019,False
4,2169fe4b-772a-45ef-9478-dbc48ee0e331,0,2024-05-09,219.20.206.172,8efe3214-ea7f-47ef-bb30-716e28f02600,(527)638-2221x33874,4908554227138584,Food,170733,False



--------------------------------------------------
Dataset Shape: (14813, 10)
--------------------------------------------------
Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14813 entries, 0 to 14812
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   transaction_id      14813 non-null  object
 1   user_id             14813 non-null  int64 
 2   transaction_time    14813 non-null  object
 3   ip                  14813 non-null  object
 4   device_id           14813 non-null  object
 5   phone_number        14813 non-null  object
 6   credit_card_number  14813 non-null  int64 
 7   order_item          14813 non-null  object
 8   amount              14813 non-null  int64 
 9   is_fraud            14813 non-null  bool  
dtypes: bool(1), int64(3), object(6)
memory usage: 1.0+ MB

--------------------------------------------------
Distribution of 'is_fraud' label:


,Proportion,proportion
0,False,0.982245
1,True,0.017755


--------------------------------------------------


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix

# Simple GNN Model
class FraudGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super(FraudGNN, self).__init__()
        self.conv1 = nn.Linear(input_dim, hidden_dim)
        self.conv2 = nn.Linear(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)  # Normal/Fraud

    def forward(self, x, adj):
        # First GNN layer: Aggregate neighbor information
        h1 = torch.relu(self.conv1(x))
        h1_agg = torch.mm(adj, h1)  # Average neighbors' information
        h1 = h1 + h1_agg

        # Second GNN layer
        h2 = torch.relu(self.conv2(h1))
        h2_agg = torch.mm(adj, h2)
        h2 = h2 + h2_agg

        # Classification
        out = self.classifier(h2)
        return out

In [4]:
def create_fast_graph(df):
    """Fast graph generation using vectorized operations"""
    print("   Creating node features...")
    n = len(df)

    times = pd.to_datetime(df['transaction_time'])
    features = np.column_stack([
        df['amount'].values / 1000,
        times.dt.hour.values / 24.0,
        times.dt.weekday.values / 6.0
    ])

    print("   Computing connections...")
    row_indices = []
    col_indices = []

    for user_id in df['user_id'].unique(): #A
        user_mask = df['user_id'] == user_id
        user_indices = np.where(user_mask)[0]
        if len(user_indices) > 1:
            for i in user_indices:
                for j in user_indices:
                    if i != j:
                        row_indices.append(i)
                        col_indices.append(j)

    for device_id in df['device_id'].unique():
        device_mask = df['device_id'] == device_id
        device_indices = np.where(device_mask)[0]
        if len(device_indices) > 1:
            for i in device_indices:
                for j in device_indices:
                    if i != j:
                        row_indices.append(i)
                        col_indices.append(j)

    print("   Creating sparse matrix...")
    if row_indices:
        data = np.ones(len(row_indices))
        adj_sparse = csr_matrix((data, (row_indices, col_indices)), shape=(n, n))

        row_sum = np.array(adj_sparse.sum(axis=1)).flatten()
        row_sum[row_sum == 0] = 1
        adj_sparse = adj_sparse.multiply(1.0 / row_sum.reshape(-1, 1)) # Normalized sparse matrix

        if n < 1000: # Original logic to convert to dense for small graphs
            adj_matrix = adj_sparse.toarray()
        else:
            adj_matrix = adj_sparse # Keep sparse for larger graphs
    else:
        adj_matrix = np.eye(n)

    return features, adj_matrix

In [5]:
def run_fast_gnn_demo(): # Function definition starts here
    """Run optimized GNN demo"""
    print("Fast GNN Fraud Detection Demo")
    print("=" * 35)

    if 'transactions_df' not in globals() or transactions_df is None or transactions_df.empty:
        print(" transactions_df not found or is empty!")
        return # Modified to just return, actual return values handled at the end of the function

    df = transactions_df
    print(f" Total data: {len(df)} transactions")

    # 1. Use full dataset (no sampling)
    print("Using full dataset")
    features, adj_matrix = create_fast_graph(df) # adj_matrix here is potentially sparse

    print(f"\n   Final nodes: {len(features)}")

    # Connection information and conversion to dense for PyTorch model
    if hasattr(adj_matrix, 'nnz'):  # If it's a SciPy sparse matrix
        print(f"   Connections (sparse): {adj_matrix.nnz}")
        if len(features) > 0:
            print(f"   Avg neighbors (sparse): {adj_matrix.nnz / len(features):.1f}")
        print("   Converting sparse adjacency to dense array for PyTorch model...")
        adj_matrix = adj_matrix.toarray() # Convert to dense NumPy array
    elif isinstance(adj_matrix, np.ndarray): # If it's already a NumPy dense array
        print(f"   Connections (dense): {(adj_matrix > 0).sum()}") # Simplified count for dense
        if len(features) > 0:
            print(f"   Avg neighbors (dense): {(adj_matrix > 0).sum(axis=1).mean():.1f}")
    else:
        # Fallback if adj_matrix is None or unexpected type from create_fast_graph
        print("   Warning: Adjacency matrix is of an unexpected type or None. Attempting to use identity matrix.")
        adj_matrix = np.eye(len(features))

    # 2. Convert to tensors
    X = torch.FloatTensor(features)
    A = torch.FloatTensor(adj_matrix)

    # 3. Create model
    model = FraudGNN(input_dim=X.shape[1], hidden_dim=16) # Use X.shape[1] for input_dim
    # The rest of the function (training and results analysis) will be in the next listing

    # --- Continuation of run_fast_gnn_demo function ---

    # 4. Train if labels exist, otherwise inference only
    if 'is_fraud' in df.columns:
        print("\n Training GNN...")
        y = torch.LongTensor(df['is_fraud'].values) # Using original df['is_fraud'].values

        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()

        best_loss = float('inf')
        patience = 0

        for epoch in range(100):
            model.train() # Set model to training mode
            optimizer.zero_grad()
            outputs = model(X, A)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            if epoch % 20 == 0: # Original condition for printing
                acc = (outputs.argmax(1) == y).float().mean()
                print(f"   Epoch {epoch}: Loss={loss.item():.3f}, Acc={acc.item():.3f}") # Using .item() for scalar loss

                if loss.item() < best_loss: # Early stopping based on training loss
                    best_loss = loss.item()
                    patience = 0
                else:
                    patience += 1
                    if patience > 3 and epoch > 40:
                        print(f"   Early stopping at epoch {epoch}")
                        break
        # Removed "GNN training finished." from here to match original single print at the end of function

    else: # This else block was part of the original code for the 'is_fraud' in df.columns check
        print("\n GNN inference mode...")

    # The function continues in Listing 9.11 with results analysis and return...

    # --- Continuation of run_fast_gnn_demo function ---

    # 5. Results analysis (This part is reached after training or if in inference mode)
    model.eval()
    with torch.no_grad():
        outputs = model(X, A)
        fraud_probs = F.softmax(outputs, dim=1)[:, 1]

    top_indices = fraud_probs.argsort(descending=True)[:5]
    print(f"\n Most Suspicious Transactions TOP 5:")
    for i, idx_loop_var in enumerate(top_indices): # Renamed idx to idx_loop_var to avoid conflict if idx was used before
        idx_int = int(idx_loop_var.item()) # Original: idx = int(idx)
        row = df.iloc[idx_int]       # Original: row = df.iloc[idx]
        prob_val = fraud_probs[idx_int].item() # Original: prob = fraud_probs[idx]

        actual_str = f" (Actual: {'Fraud' if 'is_fraud' in df.columns and row['is_fraud'] else 'Normal'})" if 'is_fraud' in df.columns else ""
        print(f"   {i+1}. User:{row['user_id']}, Amount:{row['amount']}, Prob:{prob_val:.3f}{actual_str}")

    print(f"\n Optimized GNN analysis complete!")

    return model, fraud_probs
# --- End of run_fast_gnn_demo function definition ---

# Now, actually run the demo by calling the fully defined function
if 'transactions_df' in globals() and transactions_df is not None:
    trained_model_final_output, final_fraud_probs_gnn_output = run_fast_gnn_demo() # Call the function
else:
    print("Cannot run GNN demo as 'transactions_df' is not loaded or is empty.")

Fast GNN Fraud Detection Demo
 Total data: 14813 transactions
Using full dataset
   Creating node features...
   Computing connections...


   Creating sparse matrix...

   Final nodes: 14813
   Connections (sparse): 60142
   Avg neighbors (sparse): 4.1
   Converting sparse adjacency to dense array for PyTorch model...



 Training GNN...


   Epoch 0: Loss=149.684, Acc=0.018


   Epoch 20: Loss=1.991, Acc=0.982


   Epoch 40: Loss=2.251, Acc=0.982


   Epoch 60: Loss=0.833, Acc=0.982


   Epoch 80: Loss=1.261, Acc=0.982



 Most Suspicious Transactions TOP 5:
   1. User:205, Amount:12168, Prob:0.207 (Actual: Fraud)
   2. User:1550, Amount:19086, Prob:0.122 (Actual: Fraud)
   3. User:163, Amount:57917, Prob:0.005 (Actual: Fraud)
   4. User:2498, Amount:65591, Prob:0.002 (Actual: Fraud)
   5. User:2538, Amount:9576, Prob:0.000 (Actual: Normal)

 Optimized GNN analysis complete!
